# An API for Linear Dynamical Systems

In previous posts, I worked out the Kalman filter and some of the standard inference algorithms for a Gaussian linear dynamical system (LDS). This post takes a step back and summarizes the key functions that a library for linear dynamical systems would need to include. The code listings will be in Python, but the ideas should be easy to apply to other languages.

<!-- TEASER_END -->

## Top-Level Functions

We first describe top-level functions that reason at an abstraction layer including systems, inputs, outputs, and beliefs.

In [2]:
def simulate(S, m, C, U):
    n_steps = len(U)
    X = np.zeros((n_steps, n_hidden(S)))
    for i, u in enumerate(U):
        X[i] = mvn(m, C)
        m, C = advance(*dyn_model(S), X[i], u)
    return X

def kfilter(S, m, C, U, Y):
    score = 0.0
    for u, y in zip(U, Y):
        m, C, logprob = correct(*obs_model(S), m, C, y)
        m, C = predict(*dyn_model(S), m, C, u)
        score += logprob
    return m, C, score

def smooth(S, m, C, U, Y):
    n_steps = len(U)
    alpha = _forward(S, m, C, U, Y)
    X = np.zeros((n_steps, n_hidden(S)))
    
    X[-1] = mvn(*alpha[-1])
    for i in range(-2, -(n_steps + 1), -1):
        X[i] = reverse(*dyn_model(S), *alpha[i], U[i], X[i + 1])
        
    return X

def _forward(S, m, C, U, Y):
    alpha = []
    for u, y in zip(U, Y):
        m, C, _ = correct(*obs_model(S), m, C, y)
        alpha.append((m, C))
        m, C = predict(*dyn_model(S), m, C, u)
    return alpha

# Mid-Level Functions

The top-level functions depend on a number of mid-level functions. There are

In [3]:
def advance(F, G, Q, u, x):
    pass

def reverse(F, G, Q, m, C, u, x):
    pass

def predict(F, G, Q, m, C, u):
    pass

def correct(H, R, m, C, y):
    pass

# Low-Level Functions

These are the implementations of low-level mathematical operations that support the mid-level functions.